In this notebook the HDBSCAN algorithm is runned on the latant features and the full dataset.

# 00 - General Operationss

In [1]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd 

from sklearn.cluster import HDBSCAN
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import PowerTransformer

import tensorflow as tf

import sys
from pathlib import Path
sys.path.append(str(Path("../../src/utils").resolve()))
from dataset_balancing import balance_dataset
sys.path.append(str(Path("../../src/models").resolve()))
from autoencoder import AutoEncoder

Set the random seed for reproducibility:

In [2]:
random_state = 42
np.random.seed(random_state)

In [3]:
latent_space_dim = 10
hidden_dims = (64, 32, latent_space_dim)
# batch_size = 256

n_samples_min = 2000

families_dict = {
    # Neutral_Background
    "AlbedoPhotons" : "Neutral_Background",
    "AtmosphericNeutrons" : "Neutral_Background",
    "CosmicPhotons" : "Neutral_Background",

    # TEMP: tgf
    "TGF": "TGF",
    
    # EM_Primary 
    "PrimaryElectrons" : "EM_Primary",
    "PrimaryPositrons" : "EM_Primary",

    # EM_Secondary
    "SecondaryElectrons" : "EM_Secondary",
    "SecondaryPositrons" : "EM_Secondary",

    # Hadronic_Heavy
    "PrimaryProtons" : "Hadronic_Heavy",
    "PrimaryAlphas" : "Hadronic_Heavy",
    "SecondaryProtonsUpward" : "Hadronic_Heavy",
    "SecondaryProtonsDownward" : "Hadronic_Heavy"
}
FAMILY_CLUSTERING = True

corr_threshold = 0.90

test_size = 0.2

lr = 1e-3

# 01 - Data pre-processing

In this section data are loaded and balanced. Feature higly correlated are dropped in the dataset. 

## Data Loading

In [4]:
file_path = "../../data/Engineered_Datasets/"
ds = pd.read_csv(file_path + "bgtgf_ge3hit_dataset.csv")
print(f"Combined dataset shape before balancing: {ds.shape}")
print(ds['particle_type'].value_counts())

Combined dataset shape before balancing: (224816, 44)
particle_type
TGF                         141183
SecondaryPositrons           17141
PrimaryPositrons             14546
PrimaryProtons               12455
AlbedoPhotons                 8789
PrimaryAlphas                 7215
PrimaryElectrons              6464
SecondaryProtonsUpward        4594
SecondaryProtonsDownward      4256
SecondaryElectrons            4205
AtmosphericNeutrons           2043
CosmicPhotons                 1925
Name: count, dtype: int64


In [5]:
n_samples = min(n_samples_min, ds['particle_type'].value_counts().min())
print(f"Number of samples per class: {n_samples}")

ds = balance_dataset(ds, target_col="particle_type", n_samples = n_samples, random_state=random_state)
print(f"Dataset shape after balancing: {ds.shape}")
ds["particle_type"].value_counts()

if FAMILY_CLUSTERING:
    ds["particle_family"] = ds["particle_type"].map(families_dict)
    #print(f"particle_family value counts:\n{ds['particle_family'].value_counts()}")

    n_samples = ds['particle_family'].value_counts().min()
    ds = balance_dataset(ds, target_col="particle_family", n_samples = n_samples, random_state=random_state)
    print(f"\nDataset shape after balancing families: {ds.shape}")
    print(f"particle_family value counts after balancing:\n{ds['particle_family'].value_counts()}")

Number of samples per class: 1925
Dataset shape after balancing: (23100, 44)

Dataset shape after balancing families: (9625, 45)
particle_family value counts after balancing:
particle_family
EM_Primary            1925
EM_Secondary          1925
Hadronic_Heavy        1925
Neutral_Background    1925
TGF                   1925
Name: count, dtype: int64


## Dropping Feature

Now drop feature higly correlated: 

In [6]:
drop_columns = ["pixel_ijk", "particle_type"]
if FAMILY_CLUSTERING:
    drop_columns.append("particle_family")
ds_drop = ds.drop(columns=drop_columns)


# correlation_matrix = ds_drop.corr(method='pearson')
# upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
# cols_to_drop = [
#     column
#     for column in upper_triangle.columns
#     if any(upper_triangle[column] > corr_threshold)
# ]
# print(f"Columns to drop: {len(cols_to_drop)}\n{cols_to_drop}")


In [ ]:
# cols_to_drop.remove("L_max_metric") if "L_max_metric" in cols_to_drop else None
# cols_to_drop.remove("N_b") if "N_b" in cols_to_drop else None
# cols_to_drop.remove("L_skel") if "L_skel" in cols_to_drop else None
# print(f"Columns to drop after removing L_max_metric, N_b, and L_skel: {len(cols_to_drop)}\n{cols_to_drop}")

# ds_drop = ds_drop.drop(columns=cols_to_drop)
# print(f"Dataset shape after dropping highly correlated features: {ds_drop.shape}")

Columns to drop after removing L_max_metric, N_b, and L_skel: 9
['E_max', 'E_p10', 'E_p25', 'E_p90', 'E_p99', 'E_iqr', 'E_kurt', 'frac_top1', 'lam1']
Dataset shape after dropping highly correlated features: (9625, 33)


## Data Conversion

Split data in train and test set:

In [7]:
if FAMILY_CLUSTERING:
    Xtrain, Xtest, ytrain, ytest = train_test_split(ds_drop, ds['particle_family'], test_size=test_size, random_state=random_state)
else:
    Xtrain, Xtest, ytrain, ytest = train_test_split(ds_drop, ds['particle_type'], test_size=test_size, random_state=random_state)


print(f"Xtrain shape: {Xtrain.shape}, ytrain shape: {ytrain.shape}")
print(ytrain.value_counts())

Xtrain shape: (7700, 42), ytrain shape: (7700,)
particle_family
Neutral_Background    1556
EM_Secondary          1555
Hadronic_Heavy        1536
EM_Primary            1530
TGF                   1523
Name: count, dtype: int64


Normalize data to have mean 0 and variance 1:

In [8]:
pt = PowerTransformer(standardize=True)
Xtrain_transformed = pt.fit_transform(Xtrain)
Xtest_transformed = pt.transform(Xtest)

Convert to tensors and use dataloader to load batch of data during training cycle:

In [9]:
train_tensor = tf.convert_to_tensor(Xtrain_transformed, dtype=tf.float32)

train_loader = (
    tf.data.Dataset.from_tensor_slices(train_tensor)
    .shuffle(buffer_size=1024, seed=random_state)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE) 
)

# 02 - Model training

In this section the AE is trained and the best weights are kept for model *inference*:

In [10]:
n_input_features = len(Xtrain.columns)

model = AutoEncoder(
    input_dim=n_input_features,
    hidden_layers=hidden_dims,
    dropout_rate=0.0,
    activation="leaky_relu",
    output_activation=None,
    random_state=random_state,
)

lr = 1e-3
optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
mse_criterion = tf.keras.losses.MeanSquaredError()


In [11]:
number_of_epochs = 300
best_loss = np.inf
best_weights = None

@tf.function
def train_step(batch_x):
    with tf.GradientTape() as t:
        output = model(batch_x, training=True)[1]
        loss_val = mse_criterion(batch_x, output)

    # Gradient computation and weight update
    gradients = t.gradient(loss_val, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss_val


loss_history = []
for epoch in tqdm(range(number_of_epochs), desc="Training Progress"):
    
    for batch_x in train_loader:
        loss_val = train_step(batch_x)
        loss_history.append(loss_val.numpy())

    mean_loss = float(np.mean(loss_history[-len(train_loader):]))  # Calculate mean loss for the current epoch
    if mean_loss < best_loss:
        best_loss = mean_loss
        best_weights = model.get_weights()         

model.set_weights(best_weights)
print(f"Best loss: {best_loss}")

Training Progress:   0%|          | 0/300 [00:00<?, ?it/s]

Training Progress: 100%|██████████| 300/300 [00:18<00:00, 15.84it/s]

Best loss: 0.06820379197597504


# 03 - Clustering

In [12]:
if FAMILY_CLUSTERING:
    title_suffix = "Particle Family"
else:
    title_suffix = "Particle Type"

In [13]:
def plot_confusion_matrix(y_true, y_pred, title_suffix, clsut_algor_algo="KMeans"):
    ct = pd.crosstab(y_pred, y_true, normalize="index") * 100

    plt.figure(figsize=(10, 4.5))
    sns.heatmap(ct, annot=True, fmt=".1f", cmap="viridis")
    plt.title(
        f"AutoEncoder + {clsut_algor_algo}: Cluster vs {title_suffix} (%)",
        fontweight="bold",
    )
    plt.xlabel("Real Particle Type")
    plt.ylabel("Assigned Cluster (AE Latent)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [15]:
params = {
    "min_cluster_size" : [10, 20, 30, 40, 50],
    "min_samples" : [5, 10, 15, 20, 25, 30],
    "cluster_selection_epsilon" : [0.3, 0.5, 0.7, 1.0],
    'cluster_selection_method': ['eom', 'leaf'],

    #"copy" : True,
    #"n_jobs" : [-1],
}

pg = list(ParameterGrid(params))

## 03.1 - HDBSCAN on Latent Space Features

In [19]:
# x_test_t = tf.convert_to_tensor(Xtest_transformed, dtype=tf.float32)
# z_embedded = model(x_test_t, training=False)[0]

x_train_t = tf.convert_to_tensor(Xtrain_transformed, dtype=tf.float32)
z_embedded = model(x_train_t, training=False)[0]

In [21]:
rows = []
for cfg in tqdm(ParameterGrid(params), desc="HDBSCAN Parameter Grid Search"):
    clusterer = HDBSCAN(copy=True, n_jobs=-1, **cfg)
    labels = clusterer.fit_predict(z_embedded)
    noise = (labels == -1)
    n_clusters = len(set(labels) - {-1})

    row = {
        **cfg, 
        "n_clusters": n_clusters, 
        "noise_%": round(noise.mean() * 100, 1)
    }
    
    if n_clusters >= 2 and (~noise).sum() > 1:
        y_clustered = ytrain.to_numpy()[~noise]
        row["AMI"] = normalized_mutual_info_score(y_clustered, labels[~noise])
        row["ARI"] = adjusted_rand_score(y_clustered, labels[~noise])
    rows.append(row)

df_hdb_results = pd.DataFrame(rows)
print(df_hdb_results[["n_clusters", "noise_%"]].describe())

HDBSCAN Parameter Grid Search: 100%|██████████| 240/240 [01:16<00:00,  3.14it/s]

       n_clusters     noise_%
count  240.000000  240.000000
mean     2.783333   86.166667
std      1.774686    8.397821
min      0.000000   40.600000
25%      2.000000   82.400000
50%      2.500000   84.900000
75%      3.000000   90.400000
max     11.000000  100.000000


In [ ]:
valid_models = df_hdb_results[
    (df_hdb_results['n_clusters'].between(4, 7))
    & (df_hdb_results['noise_%'] <= 30.0)
].sort_values(by='ARI', ascending=False)

print('Best configurations found:')
print(valid_models.head(10))

Best configurations found:
Empty DataFrame
Columns: [cluster_selection_epsilon, cluster_selection_method, min_cluster_size, min_samples, n_clusters, noise_%, AMI, ARI]
Index: []


As we can see from the results above, the clustering algorithm is not able to identify any useful cluster.

## 03.2 - HDBSCAN on full feature set

In [25]:
rows = []
for cfg in tqdm(ParameterGrid(params), desc="HDBSCAN Parameter Grid Search"):
    clusterer = HDBSCAN(copy=True, n_jobs=-1, **cfg)
    labels = clusterer.fit_predict(Xtrain_transformed)
    noise = (labels == -1)
    n_clusters = len(set(labels) - {-1})

    row = {
        **cfg, 
        "n_clusters": n_clusters, 
        "noise_%": round(noise.mean() * 100, 1)
    }
    
    if n_clusters >= 2 and (~noise).sum() > 1:
        y_clustered = ytrain.to_numpy()[~noise]
        row["AMI"] = normalized_mutual_info_score(y_clustered, labels[~noise])
        row["ARI"] = adjusted_rand_score(y_clustered, labels[~noise])
    rows.append(row)

df_hdb_results = pd.DataFrame(rows)
print(df_hdb_results[["n_clusters", "noise_%"]].describe())

HDBSCAN Parameter Grid Search: 100%|██████████| 240/240 [03:47<00:00,  1.06it/s]

       n_clusters     noise_%
count  240.000000  240.000000
mean     3.266667   83.296667
std      1.734384    3.811367
min      2.000000   76.900000
25%      2.000000   79.800000
50%      3.000000   84.500000
75%      4.000000   86.400000
max     11.000000   89.200000


In [26]:
valid_models = df_hdb_results[
    (df_hdb_results['n_clusters'].between(4, 7))
    & (df_hdb_results['noise_%'] <= 30.0)
].sort_values(by='ARI', ascending=False)

print('Best configurations found:')
print(valid_models.head(10))

Best configurations found:
Empty DataFrame
Columns: [cluster_selection_epsilon, cluster_selection_method, min_cluster_size, min_samples, n_clusters, noise_%, AMI, ARI]
Index: []


Even on the full dataset the clustering algorithm gives no results. 